# Agente de Atención a Pedidos — Llama Stack + Function Calling

**Notebook complementario** (opcional) del challenge *Agente de Atención a Pedidos*,
tema **Integración Llama + WhatsApp**.

Este notebook reproduce la lógica central en Google Colab con el modelo
`llama3.1:8b`, tal como lo pide el challenge. La ejecución de referencia del
proyecto completo (servidor local) está en `resultados/` con `gemma4:12b`.

Contenido:
1. Instalar Llama Stack y cliente
2. Memoria: una conversación de Llama Stack por número de teléfono
3. Herramientas: `consultar_pedido` y `buscar_pregunta_frecuente`
4. Ciclo con function calling + medición de latencia
5. Reglas fijas (urgencia y escalamiento) antes del modelo
6. Prueba de 4 mensajes

## 1. Instalar dependencias

Llama Stack **0.7.3** con su cliente **0.7.5** (versiones probadas; la CLI
antigua `llama stack serve` ya no existe, se usa `llama stack run`).

In [ ]:
!pip install -q "llama-stack==0.7.3" "llama-stack-client==0.7.5" "mcp<1.13" "pydantic<2.12" requests

### Levantar el servidor

En Colab se registra un proveedor de inferencia (Ollama local con `!ollama pull llama3.1:8b`,
o el proveedor que esté disponible) y se corre `llama stack run` en segundo plano.
En el proyecto local el equivalente es `llama stack run llama_stack/run.yaml --port 8321`.

In [ ]:
import os, subprocess, time, urllib.request

BASE_URL = "http://127.0.0.1:8321"

# Si el servidor local no está corriendo, se levanta con run.yaml
try:
    urllib.request.urlopen(BASE_URL + "/v1/health", timeout=3)
    print("Llama Stack ya está corriendo")
except Exception:
    subprocess.Popen(["llama", "stack", "run", "llama_stack/run.yaml", "--port", "8321"])
    for _ in range(40):
        time.sleep(2)
        try:
            urllib.request.urlopen(BASE_URL + "/v1/health", timeout=3)
            print("Llama Stack listo")
            break
        except Exception:
            pass

from llama_stack_client import LlamaStackClient
cliente = LlamaStackClient(base_url=BASE_URL)
MODELO = "ollama/llama3.1:8b"   # modelo que exige el challenge

## 2. Memoria: una conversación de Llama Stack por número de teléfono

Sin esto, Llama trataría cada mensaje como si fuera el primero. Cada teléfono
tiene su `conversation_id` **y su lock**, para serializar los mensajes del
mismo usuario (evita respuestas fuera de orden).

In [ ]:
import threading, re
from typing import Dict

def normalizar_numero_mx(valor: str) -> str:
    """Deja solo digitos y agrega el 52 de Mexico si falta."""
    digitos = re.sub(r"\D", "", valor)
    if digitos.startswith("52") and len(digitos) == 12:
        return digitos
    return "52" + digitos[-10:]

class Memoria:
    def __init__(self, client):
        self.cliente = client
        self.sesiones: Dict[str, dict] = {}

    def sesion(self, telefono: str) -> dict:
        if telefono not in self.sesiones:
            conv = self.cliente.conversations.create(metadata={"telefono": telefono})
            self.sesiones[telefono] = {
                "conversation_id": conv.id,
                "lock": threading.Lock(),
                "turnos": 0,
            }
        return self.sesiones[telefono]

memoria = Memoria(cliente)
s = memoria.sesion("5215511110009")
print("conversation_id para 5215511110009 ->", s["conversation_id"])

## 3. Las herramientas: contrato (esquema) e implementación (funciones)

Hasta aquí Llama solo generaba texto a partir de texto: si preguntaban por un
pedido, podía inventar un estatus que sonara razonable. **Function calling**
le describe una función real de tu código (nombre, qué hace, qué parámetros
necesita) y el modelo, cuando lo considera, pide que se ejecute en vez de
responder directamente.

In [ ]:
import json, re

PEDIDOS = {
    "9988": {"numero_pedido": 9988, "estado": "en_transito", "transportista": "Estafeta",
             "guia": "7123456789", "fecha_estimada": "2026-11-05",
             "ultimo_movimiento": "2026-11-01: paquete en centro de distribucion Guadalajara"},
    "4521": {"numero_pedido": 4521, "estado": "entregado", "transportista": "DHL",
             "guia": "9988776655", "fecha_estimada": "2026-10-28",
             "ultimo_movimiento": "2026-10-28: entregado en recepcion"},
}

FAQ = {
    "pagos": "Aceptamos tarjeta de credito y debito (Visa, Mastercard, Amex), transferencia SPEI y PayPal.",
    "envios": "Envios a todo Mexico en 3 a 7 dias habiles segun la ciudad.",
    "garantia": "Todos los productos tienen 12 meses de garantia por defecto de fabricacion.",
}

ESQUEMAS = [
    {"type": "function", "name": "consultar_pedido",
     "description": ("Recibe un numero_pedido entero y devuelve el estatus real del pedido: "
                     "estado, transportista, guia y fecha estimada. Usala EXCLUSIVAMENTE cuando "
                     "el cliente pregunte por el estado o rastreo de un pedido concreto."),
     "parameters": {"type": "object",
                    "properties": {"numero_pedido": {"type": "integer",
                                                      "description": "Numero de pedido, por ejemplo 9988"}},
                    "required": ["numero_pedido"]}},
    {"type": "function", "name": "buscar_pregunta_frecuente",
     "description": ("Busca la respuesta oficial a dudas generales (pagos, envios, garantia, "
                     "horarios). Recibe un tema. No la uses para estatus de pedidos."),
     "parameters": {"type": "object",
                    "properties": {"tema": {"type": "string", "description": "pagos, envios o garantia"}},
                    "required": ["tema"]}},
]

def consultar_pedido(numero_pedido: int) -> dict:
    p = PEDIDOS.get(str(int(numero_pedido)))
    return p if p else {"ok": False, "error": f"No existe el pedido {numero_pedido}."}

def buscar_pregunta_frecuente(tema: str) -> dict:
    clave = re.sub(r"[^a-z]", "", tema.lower())
    for k, v in FAQ.items():
        if clave in k or k in clave:
            return {"ok": True, "tema": k, "respuesta": v}
    return {"ok": False, "error": f"No hay FAQ para '{tema}'."}

def validar_y_ejecutar(nombre, argumentos):
    """Rechaza argumentos invalidos ANTES de tocar los datos."""
    if nombre == "consultar_pedido":
        v = argumentos.get("numero_pedido")
        if isinstance(v, str) and v.isdigit(): v = int(v)
        if not isinstance(v, int) or v <= 0:
            return {"ok": False, "error": "argumentos invalidos: numero_pedido debe ser entero positivo"}
        return consultar_pedido(v)
    if nombre == "buscar_pregunta_frecuente":
        t = argumentos.get("tema")
        if not isinstance(t, str) or not t.strip():
            return {"ok": False, "error": "argumentos invalidos: tema vacio"}
        return buscar_pregunta_frecuente(t)
    return {"ok": False, "error": f"herramienta desconocida: {nombre}"}

## 4. Generar la respuesta con historial y function calling

Se reenvía el historial reciente (la `conversation_id` conserva el contexto)
junto con las herramientas. Si `respuesta.output` trae elementos con
`type == "function_call"`, el código los **ejecuta de verdad** con los
argumentos que Llama propuso, devuelve el resultado en una segunda llamada y
solo entonces se genera la respuesta final en lenguaje natural.

Se mide la latencia del ciclo por fases (razonamiento / acción), que es lo
único que percibe el usuario.

In [ ]:
INSTRUCCIONES = (
    "Eres el asistente de atencion a pedidos de la tienda Mercado Central. "
    "Responde en espanol, maximo 3 lineas. Usa consultar_pedido para estado de "
    "pedidos y buscar_pregunta_frecuente para dudas generales; nunca inventes "
    "estatus ni fechas."
)

def responder(telefono: str, mensaje: str) -> dict:
    sesion = memoria.sesion(normalizar_numero_mx(telefono))
    with sesion["lock"]:
        t0 = time.perf_counter()
        entradas = [{"role": "user", "content": mensaje}]
        tools_llamadas, texto, ms_raz, ms_acc = [], "", 0, 0

        for _ in range(2):
            a = time.perf_counter()
            r = cliente.responses.create(
                model=MODELO,
                conversation=sesion["conversation_id"],
                instructions=INSTRUCCIONES,
                input=entradas,
                tools=ESQUEMAS,
                max_output_tokens=220,
                temperature=0.3,
            )
            ms_raz += int((time.perf_counter() - a) * 1000)

            llamadas = [i for i in r.output if i.type == "function_call"]
            if not llamadas:
                texto = (r.output_text or "").strip()
                break

            for lam in llamadas:
                d = lam.model_dump()
                args = d.get("arguments") or "{}"
                if isinstance(args, str): args = json.loads(args)
                b = time.perf_counter()
                resultado = validar_y_ejecutar(d.get("name"), args)
                ms_acc += int((time.perf_counter() - b) * 1000)
                tools_llamadas.append({"nombre": d.get("name"), "argumentos": args, "ok": resultado.get("ok")})
                entradas.append({"type": "function_call", "call_id": d.get("call_id"),
                                 "name": d.get("name"), "arguments": d.get("arguments")})
                entradas.append({"type": "function_call_output", "call_id": d.get("call_id"),
                                 "output": json.dumps(resultado, ensure_ascii=False)})

        sesion["turnos"] += 1
        return {"mensaje": mensaje, "respuesta": texto, "tools": tools_llamadas,
                "razonamiento_ms": ms_raz, "accion_ms": ms_acc,
                "ciclo_ms": int((time.perf_counter() - t0) * 1000)}

## 5. Reglas fijas: urgencia y escalamiento SIN pasar por Llama

Dos reglas inquebrantables que se evalúan **antes** del modelo. Es el mismo
principio que en el challenge anterior (la de salud animal se sustituye por
la de **cargo duplicado**, que es la urgencia que exige este dominio).

In [ ]:
REGLAS = [
    ("urgencia_fraude",
     ["cargo duplicado", "me cobraron dos veces", "cobro doble", "fraude", "cargo no reconocido"],
     "🚨 Duplicado detectado: revisamos ese cargo de inmediato. No vuelvas a intentar el pago; "
     "un agente verificara el cobro y, si procede, se reversara en 24-48 h."),
    ("escalado_humano",
     ["quiero hablar con", "hablar con una persona", "quiero un asesor", "me quiero quejar"],
     "Claro, te paso con una persona del equipo de atencion. Ya registre tu solicitud y "
     "te contactamos en breve por este mismo numero."),
]

def regla_fija(texto: str):
    limpio = texto.lower()
    for nombre, patrones, mensaje in REGLAS:
        if any(p in limpio for p in patrones):
            return {"nombre": nombre, "respuesta": mensaje}
    return None

def atender(telefono: str, mensaje: str) -> dict:
    t0 = time.perf_counter()
    r = regla_fija(mensaje)
    if r:
        return {"fuente": "regla", "regla": r["nombre"], "respuesta": r["respuesta"],
                "ciclo_ms": int((time.perf_counter() - t0) * 1000)}
    t = responder(telefono, mensaje)
    t["fuente"] = "llama-stack"
    return t

## 6. Prueba: 4 mensajes que demuestran las 4 competencias

In [ ]:
TEL = "5215511110009"

mensajes = [
    "Donde esta mi pedido 9988?",              # function calling: consultar_pedido
    "Y en que fecha llega?",                   # memoria: contesta sin repetir el numero
    "Me cobraron dos veces, hay un cargo duplicado!",  # regla de urgencia, sin Llama
    "Quiero hablar con una persona",            # regla de escalamiento, sin Llama
]

for i, m in enumerate(mensajes, 1):
    t = atender(TEL, m)
    print(f"--- mensaje {i} [{t['fuente']}] ciclo {t['ciclo_ms']} ms ---")
    print("usuario:", m)
    print("agente :", t["respuesta"][:200])
    for h in t.get("tools", []):
        print("  tool :", h["nombre"], h["argumentos"], "OK" if h["ok"] else "RECHAZADA")
    if t.get("regla"):
        print("  regla:", t["regla"], "(sin llamada al modelo)")
    print()

### Qué se demostró

- **Function calling**: el modelo pidió `consultar_pedido` en vez de inventar el estatus.
- **Integración de resultados**: la respuesta final usa los datos devueltos por la función.
- **Memoria**: el follow-up se entiende por la misma `conversation_id`.
- **Reglas fijas**: urgencia y escalamiento responden en milisegundos sin tocar el modelo.

La prueba ejecutada del proyecto completo (webhook, recorte de historial, mock de
Graph API y veredicto 6/6) está en `resultados/`; este notebook es la versión
compacta de la lógica con `llama3.1:8b`.